In [1]:
import os
import re

In [2]:
docs_md_path = os.path.abspath("../documents/processed")
docs_md_cats  =os.listdir(docs_md_path)
print(f"Categories: {docs_md_cats}")

Categories: ['workout', 'nutrition']


In [3]:
def remove_pdf_noise(text: str) -> str:
    noise_patterns = [
        r"Downloaded from.*",
        r"Unauthorized reproduction.*",
        r"Copyright.*",
        r"DOI:\s*\S+",
        r"Author Manuscript.*",
        r"HHS Public Access.*",
    ]

    for pattern in noise_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    text = re.sub(r"\n.*\|\s*DOI:.*\n", "\n", text)

    text = re.sub(r"Page\s*\d+(\s*of\s*\d+)?", "", text, flags=re.IGNORECASE)

    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)

    text = re.sub(r"\n[-–—]*\s*\d+\s*[-–—]*\n", "\n", text)

    text = re.sub(r"-\n", "", text)

    return text


def merge_lines(text: str) -> str:
    """
    Merge single line breaks into spaces (keep paragraphs)
    """
    return re.sub(r"(?<!\n)\n(?!\n)", " ", text)


def normalize_spaces(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" ?([.,;:!?])", r"\1", text)
    return text


def normalize_empty_lines(text: str) -> str:
    # strip spaces per line
    text = "\n".join(line.strip() for line in text.splitlines())

    # max one empty line
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text

def remove_unnecessary_sections(text: str) -> str:
    text = re.sub(
        r"(ABSTRACT|Abs\s*Tr\s*ACT|SUMMARY).*?(?=\n\s*(INTRODUCTION|In\s*T\s*rodu\s*CTI\s*on|Background|CHAPTER ONE|Results)\b)",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    text = re.sub(
        r"(KEYWORDS|Key Words)\s*:?.*?(?=\n\s*(INTRODUCTION|Background|CHAPTER|Results)\b)",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    text = re.sub(
        r"\n\s*(REFERENCES|References|BIBLIOGRAPHY|Bibliography)\b.*",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    text = re.sub(
        r"Table of Contents.*?(?=\n\s*CHAPTER ONE\b|\n\s*Chapter 1\b)",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    text = re.sub(
        r"Letter From the Founder.*?(?=\n\s*CHAPTER ONE\b|\n\s*Chapter 1\b)",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    return text


def clean_text(text: str) -> str:
    text = remove_pdf_noise(text)
    text = merge_lines(text)       
    text = normalize_spaces(text)
    text = remove_unnecessary_sections(text)
    text = normalize_empty_lines(text)
    return text.strip()

In [4]:
docs_md_cleaned_path = os.path.abspath("../documents/cleaned")
os.makedirs(docs_md_cleaned_path, exist_ok=True)

for cat in docs_md_cats:
    cat_path = os.path.join(docs_md_path, cat)
    cat_cleaned_path = os.path.join(docs_md_cleaned_path, cat)
    os.makedirs(cat_cleaned_path, exist_ok=True)
    for doc in os.listdir(cat_path):
        doc_path = os.path.join(cat_path, doc)
        with open(doc_path, "r", encoding="utf-8") as f:
            text = f.read()
        cleaned_text = clean_text(text)
        cleaned_doc_path = os.path.join(cat_cleaned_path, doc)
        with open(cleaned_doc_path, "w", encoding="utf-8") as f:
            f.write(cleaned_text)
            print(f"doc : {doc} cleaned and saved.")
            print(f"Original length: {len(text)}, Cleaned length: {len(cleaned_text)}")

doc : the_mechanisms_of_muscle_hypertrophy_and_their.40.md cleaned and saved.
Original length: 103192, Cleaned length: 100994
doc : progression_models_in_resistance_training_for.26.md cleaned and saved.
Original length: 140643, Cleaned length: 137093
doc : basics_of_strength_and_conditioning_manual.md cleaned and saved.
Original length: 230257, Cleaned length: 225618
doc : mss-51-094.md cleaned and saved.
Original length: 53412, Cleaned length: 52949
doc : nihms851543.md cleaned and saved.
Original length: 51629, Cleaned length: 48713
doc : 12970_2017_Article_189.md cleaned and saved.
Original length: 132182, Cleaned length: 131015
doc : ncomms13103.md cleaned and saved.
Original length: 67169, Cleaned length: 66204
doc : 439.full.md cleaned and saved.
Original length: 113350, Cleaned length: 106161
